# 01 · Dataset exploration

Claims in `datasets/development_cases.jsonl` (development) and `datasets/gold_claims.jsonl` (held-out). Labels are **not** stored with claims; they come only from independent human annotations (notebook 02). This notebook reads files and never writes datasets.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
from engine.datasets import load_split, check_split_separation
from engine.notebook import banner, load_gold_labels, save_figure
dev, gold = load_split("development"), load_split("gold")
df = pd.DataFrame([asdict(c) for c in dev + gold])
print("development claims:", len(dev), "| gold (held-out) claims:", len(gold))
df[["claim_id", "split", "company", "claim_category", "difficulty", "reporting_period", "as_of_date", "claim"]]

## Distribution by company, category and difficulty

In [ ]:
display(pd.crosstab([df.split, df.company], df.claim_category, margins=True))
display(pd.crosstab(df.split, df.difficulty.fillna("unset")))
TARGET_PER_CATEGORY = 50
progress = (df[df.split == "gold"].claim_category.value_counts()
            .reindex(["numerical", "interpretive", "ambiguous"], fill_value=0).rename("gold_claims").to_frame())
progress["target"] = TARGET_PER_CATEGORY
progress["remaining"] = (progress.target - progress.gold_claims).clip(lower=0)
progress

## Label distribution (finalized gold only)

In [ ]:
gold_labels = load_gold_labels()
if not gold_labels:
    banner("No finalized gold labels yet (needs two agreeing independent human reviewers or an adjudication). "
           "Label distribution is not shown rather than guessed.")
else:
    lab = pd.DataFrame([{"claim_id": k, "label": v["label"], "status": v["status"]} for k, v in gold_labels.items()])
    lab = lab.merge(df[["claim_id", "claim_category", "company"]], on="claim_id", how="left")
    display(pd.crosstab(lab.claim_category, lab.label, margins=True))

## Missing fields

In [ ]:
missing = pd.DataFrame({
    "reporting_period": df.reporting_period.isna(),
    "difficulty": df.difficulty.isna(),
    "relevant_filings (empty)": df.relevant_filings.apply(lambda x: not x),
    "notes": df.notes.isna(),
}).sum().rename("claims_missing").to_frame()
missing["share"] = (missing.claims_missing / max(len(df), 1)).round(2)
missing

## Evidence availability at each claim's cutoff

Eligible = SEC filings in the claim's allowed sources filed on or before `as_of_date` (the same rule the retriever applies). A missing document can be fetched with `scripts/build_corpus.py --download` (SEC EDGAR is free).

In [ ]:
from engine.corpus import eligible_filings, document_path, facts_path, RetrievalConfig
cfg_r = RetrievalConfig()
rows = []
for c in dev + gold:
    ci = c.to_input()
    fl = eligible_filings(ci.company, ci.as_of_date, ci.allowed_forms(), cfg_r.max_filings_per_form)
    cached = [os.path.exists(document_path(f["accession_number"], f["primary_document"])) for f in fl]
    rows.append({"claim_id": c.claim_id, "company": c.company, "as_of_date": c.as_of_date,
                 "eligible_filings": len(fl), "cached": sum(cached),
                 "latest_eligible": max((f["filed"] for f in fl), default=None),
                 "forms": ",".join(sorted({f["form"] for f in fl})),
                 "xbrl_facts": os.path.exists(facts_path(c.company))})
avail = pd.DataFrame(rows)
avail

## Dataset quality warnings

In [ ]:
import re
warnings = list(check_split_separation(dev, gold))
if not gold:
    warnings.append("gold_claims.jsonl is empty: held-out evaluation cannot run yet")
for cat, n in df[df.split == "gold"].claim_category.value_counts().items():
    if n < TARGET_PER_CATEGORY:
        warnings.append(f"gold category {cat}: {n}/{TARGET_PER_CATEGORY} claims")
for _, r in avail.iterrows():
    if r.eligible_filings == 0:
        warnings.append(f"{r.claim_id}: no eligible filing at cutoff {r.as_of_date}")
    elif r.cached < r.eligible_filings:
        warnings.append(f"{r.claim_id}: {r.eligible_filings - r.cached} eligible document(s) not cached")
    if not r.xbrl_facts:
        warnings.append(f"{r.claim_id}: no XBRL facts file for {r.company}")
months = "January|February|March|April|May|June|July|August|September|October|November|December"
for c in dev + gold:  # a period that ends after the cutoff cannot be verified without future data
    for m in re.finditer(rf"({months}) (\d{{1,2}}), (\d{{4}})", c.claim):
        end = pd.Timestamp(f"{m.group(1)} {m.group(2)} {m.group(3)}").date().isoformat()
        if end > c.as_of_date:
            warnings.append(f"{c.claim_id}: refers to a period ending {end}, after its cutoff {c.as_of_date}")
dupes = df[df.duplicated(["company", "claim"], keep=False)]
if len(dupes):
    warnings.append(f"duplicate claim texts: {sorted(dupes.claim_id)}")
counts = df.groupby("split").company.value_counts(normalize=True)
for (split, co), share in counts.items():
    if share > 0.6:
        warnings.append(f"{split}: {co} is {share:.0%} of claims")
print(f"{len(warnings)} warning(s)")
for w in warnings:
    print(" -", w)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
pd.crosstab(df.company, df.split).plot.bar(ax=ax[0], rot=0, title="Claims by company")
pd.crosstab(df.claim_category, df.split).plot.bar(ax=ax[1], rot=0, title="Claims by category")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
print("saved", save_figure(fig, "01_dataset_distribution"))